# Configuración y descarga de datos

In [ ]:
# =========================================================
# 1. CONFIGURACIÓN GENERAL DEL PROYECTO
# =========================================================

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sklearn.preprocessing import StandardScaler

from scipy.optimize import minimize

# -----------------------------
# Configuración reproducible
# -----------------------------
SEED = 42
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
# -----------------------------
# Universo de inversión
# -----------------------------
ASSET_TICKERS = ["SPY", "VGK", "EWJ", "VWO", "XLK", "XLE", "GLD"]
BENCHMARK_TICKER = "ACWI"
ALL_TICKERS = ASSET_TICKERS + [BENCHMARK_TICKER]


# -----------------------------
# Horizonte temporal
# -----------------------------
START_DATE = "2000-01-01"
END_DATE = "2026-05-10"   #Ultimo dia de ejecución reflejado en el trabajo# None = hasta el día de hoy


# -----------------------------
# Frecuencia y rebalanceo
# -----------------------------
PRICE_FREQ = "W-FRI"   # datos semanales, cierre del viernes
REBAL_FREQ = "M"       # rebalanceo mensual
RETURN_METHOD = "log"   # calculamos los retornos de activos originales
BACKTEST_RETURN_METHOD = "simple"   # cómo están expresados los retornos de las estrategias

# -----------------------------
# Costes y restricciones
# -----------------------------
INITIAL_CAPITAL = 10000.0
TC_BPS = 0.001            # 10 puntos básicos por unidad de turnover
MAX_TURNOVER_YEAR = 2.0   # 200% anual
ALLOW_SHORT = False # no se permite short selling.
MAX_WEIGHT = 0.35      # restricción opcional para evitar concentración excesiva
RISK_FREE_RATE = 0.03 # Bono USA 3%

print("Configuración cargada correctamente.")
print("Activos invertibles:", ASSET_TICKERS)
print("Benchmark:", BENCHMARK_TICKER)

In [ ]:
# =========================================================
# 2. DESCARGA DE DATOS
# =========================================================

def download_adjusted_prices(
    tickers: List[str],
    start_date: str,
    end_date: str | None = None,
) -> pd.DataFrame:
    """
    Descarga precios ajustados desde Yahoo Finance.

    Parameters
    ----------
    tickers : list[str]
        Lista de tickers a descargar.
    start_date : str
        Fecha de inicio en formato YYYY-MM-DD.
    end_date : str | None
        Fecha de fin en formato YYYY-MM-DD o None (Día de hoy).

    Returns
    -------
    pd.DataFrame
        DataFrame con precios ajustados por fecha.
    """
    data = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
        threads=True,
    )

    if data.empty:
        raise ValueError("No se han descargado datos. Revisa tickers o conexión.")

    # Caso 1: columnas MultiIndex
    if isinstance(data.columns, pd.MultiIndex):
        if "Close" in data.columns.get_level_values(0):
            prices = data["Close"].copy()
        elif "Close" in data.columns.get_level_values(1):
            prices = data.xs("Close", axis=1, level=1).copy()
        else:
            raise KeyError("No se encontró la columna Close en los datos descargados.")

    # Caso 2: columnas normales
    else:
        if "Close" not in data.columns:
            raise KeyError("No se encontró la columna Close en los datos descargados.")
        prices = data["Close"].copy()

        # Si solo hay un ticker, lo convertimos a DataFrame
        if isinstance(prices, pd.Series):
            prices = prices.to_frame(name=tickers[0])

    prices = prices.sort_index()
    prices.index = pd.to_datetime(prices.index)

    # Aseguramos nombres correctos si solo hay un ticker
    if len(tickers) == 1 and prices.shape[1] == 1:
        prices.columns = [tickers[0]]

    return prices


prices_daily = download_adjusted_prices(
    tickers=ALL_TICKERS,
    start_date=START_DATE,
    end_date=END_DATE
)

print("Dimensión precios diarios:", prices_daily.shape)
display(prices_daily.head())
display(prices_daily.tail())

In [ ]:
# =========================================================
# 3. DIAGNÓSTICO DE DISPONIBILIDAD TEMPORAL
# =========================================================

def first_valid_dates(df: pd.DataFrame) -> pd.Series:
    """
    Devuelve la primera fecha válida de cada columna.
    """
    return pd.Series({col: df[col].first_valid_index() for col in df.columns})


first_dates = first_valid_dates(prices_daily)

print("Primera fecha válida para cada ticker:")
display(first_dates)

common_start_date = first_dates.max()
print(f"\nFecha común efectiva para cartera: {common_start_date.date()}")

In [ ]:
# =========================================================
# 4. RECORTE PERIODO COMÚN & LIMPIEZA BÁSICA
# =========================================================
# Creamos espacio común
prices_common = prices_daily.loc[common_start_date:].copy()

# Eliminamos filas completamente vacías
prices_common = prices_common.dropna(how="all")

# Forward fill moderado para pequeños huecos de mercado (días festivos, problemas de descarga, huecos raros, ...)
prices_common = prices_common.ffill()

# Si todavía quedan NaNs al inicio, se eliminan
prices_common = prices_common.dropna()

print("Dimensión precios periodo común:", prices_common.shape)
print("¿Quedan NaN?:", prices_common.isna().sum().sum())
display(prices_common.head())

In [ ]:
# =========================================================
# 5. CONVERSIÓN A FRECUENCIA SEMANAL
# =========================================================

prices_weekly = prices_common.resample(PRICE_FREQ).last().dropna(how="all")
prices_weekly = prices_weekly.ffill().dropna()

print("Dimensión precios semanales:", prices_weekly.shape)
display(prices_weekly.head())
display(prices_weekly.tail())

In [ ]:
# =========================================================
# 6. SEPARACIÓN ACTIVOS / BENCHMARK
# =========================================================

asset_prices = prices_weekly[ASSET_TICKERS].copy()
benchmark_prices = prices_weekly[[BENCHMARK_TICKER]].copy()

print("Activos invertibles:", asset_prices.columns.tolist())
print("Benchmark:", benchmark_prices.columns.tolist())
print("Rango temporal semanal:",
      asset_prices.index.min().date(), "->", asset_prices.index.max().date())

# Preprocesamiento y retornos semanales

In [ ]:
# =========================================================
# 7. CÁLCULO DE RETORNOS SEMANALES
# =========================================================

def compute_returns(prices: pd.DataFrame, method: str = "simple") -> pd.DataFrame:
    """
    Calcula retornos a partir de los precios.

    Parameters
    ----------
    prices : pd.DataFrame
        DataFrame de precios.
    method : str
        "simple" para retornos simples o "log" para retornos logarítmicos.

    Returns
    -------
    pd.DataFrame
        DataFrame de retornos.
    """
    if method == "simple":
        returns = prices.pct_change()
    elif method == "log":
        returns = np.log(prices / prices.shift(1))
    else:
        raise ValueError("method debe ser 'simple' o 'log'.")

    return returns.dropna(how="all")


# Retornos logaritmicos semanales
returns_weekly = compute_returns(prices_weekly, method=RETURN_METHOD)

asset_returns = returns_weekly[ASSET_TICKERS].copy()
benchmark_returns = returns_weekly[[BENCHMARK_TICKER]].copy()

print("Dimensión retornos semanales:", returns_weekly.shape)
print("Rango temporal retornos:",
      returns_weekly.index.min().date(), "->", returns_weekly.index.max().date())

display(asset_returns.head())
display(benchmark_returns.head())

In [ ]:
# =========================================================
# 8. CONTROL DE CALIDAD DE RETORNOS
# =========================================================

def check_data_quality(df: pd.DataFrame, name: str) -> None:
    """
    Muestra diagnóstico básico de calidad de datos.
    """
    print(f"--- {name} ---")
    print("Filas:", df.shape[0])
    print("Columnas:", df.shape[1])
    print("NaN totales:", df.isna().sum().sum())
    print("Infinitos totales:", np.isinf(df.values).sum())
    print()


check_data_quality(asset_returns, "Retornos activos")
check_data_quality(benchmark_returns, "Retornos benchmark")

# Sustituimos infinitos si existiesen y eliminamos filas con NaN
asset_returns = asset_returns.replace([np.inf, -np.inf], np.nan).dropna()
benchmark_returns = benchmark_returns.replace([np.inf, -np.inf], np.nan).dropna()

# Alineamos activos y benchmark
common_index = asset_returns.index.intersection(benchmark_returns.index)

asset_returns = asset_returns.loc[common_index]
benchmark_returns = benchmark_returns.loc[common_index]

print("Tras limpieza y alineación:")
check_data_quality(asset_returns, "Retornos activos")
check_data_quality(benchmark_returns, "Retornos benchmark")

In [ ]:
# =========================================================
# 9. ESTADÍSTICOS DESCRIPTIVOS DE RETORNOS SEMANALES
# =========================================================
#ASSETS
summary_assets = pd.DataFrame({
    "media_semanal": asset_returns.mean(),
    "vol_semanal": asset_returns.std(),
    "min_semanal": asset_returns.min(),
    "max_semanal": asset_returns.max(),
    "skewness": asset_returns.skew(),
    "kurtosis": asset_returns.kurtosis()
})

summary_assets["media_anualizada_aprox"] = summary_assets["media_semanal"] * 52
summary_assets["vol_anualizada"] = summary_assets["vol_semanal"] * np.sqrt(52)

display(summary_assets.round(4))

#BENCHMARK
summary_benchmark = pd.DataFrame({
    "media_semanal": benchmark_returns.mean(),
    "vol_semanal": benchmark_returns.std(),
    "min_semanal": benchmark_returns.min(),
    "max_semanal": benchmark_returns.max(),
    "skewness": benchmark_returns.skew(),
    "kurtosis": benchmark_returns.kurtosis()
})

summary_benchmark["media_anualizada_aprox"] = summary_benchmark["media_semanal"] * 52
summary_benchmark["vol_anualizada"] = summary_benchmark["vol_semanal"] * np.sqrt(52)

display(summary_benchmark.round(4))

In [ ]:
# =========================================================
# 10. MATRIZ DE CORRELACIONES ENTRE ACTIVOS
# =========================================================

corr_matrix = asset_returns.corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlación")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
for i in range(len(corr_matrix.index)):
    for j in range(len(corr_matrix.columns)):
        plt.text(
            j, i,                                # posición (columna, fila)
            f"{corr_matrix.iloc[i, j]:.2f}",     # formato con 2 decimales
            ha="center",                         # centrado horizontal
            va="center",                         # centrado vertical
            color="white",                       # color blanco
            fontweight="bold"                    # negrita
        )

plt.title("Matriz de correlaciones de retornos semanales")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 11. CALENDARIO DE REBALANCEO MENSUAL
# =========================================================

def get_monthly_rebalance_dates(returns_index: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """
    Obtiene la última fecha semanal disponible de cada mes.
    """
    dates = pd.Series(returns_index, index=returns_index)
    rebalance_dates = dates.groupby(dates.index.to_period("M")).max()
    return pd.DatetimeIndex(rebalance_dates.values)


rebalance_dates = get_monthly_rebalance_dates(asset_returns.index)

print("Número de fechas de rebalanceo:", len(rebalance_dates))
print("Primera fecha de rebalanceo:", rebalance_dates.min().date())
print("Última fecha de rebalanceo:", rebalance_dates.max().date())

display(pd.DataFrame({"rebalance_date": rebalance_dates}).head(12))

In [ ]:
# =========================================================
# 12. COMPROBACIÓN DE PERIODICIDAD DEL REBALANCEO
# =========================================================

rebalance_diff_days = pd.Series(rebalance_dates).diff().dt.days.dropna()

print("Días entre rebalanceos:")
display(rebalance_diff_days.describe())

plt.figure(figsize=(10, 4))
plt.plot(rebalance_dates[1:], rebalance_diff_days)
plt.title("Días entre fechas de rebalanceo mensual")
plt.xlabel("Fecha")
plt.ylabel("Días")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 13. CURVAS ACUMULADAS DE ACTIVOS Y BENCHMARK
# =========================================================
def compute_cumulative_growth(
    returns: pd.Series | pd.DataFrame,
    method: str = "simple"
) -> pd.Series | pd.DataFrame:
    """
    Calcula el crecimiento acumulado de una inversión a partir de retornos.

    Parameters
    ----------
    returns : pd.Series | pd.DataFrame
        Serie o DataFrame de retornos.
    method : str
        Tipo de retorno utilizado:
        - "simple": retornos simples.
        - "log": retornos logarítmicos.

    Returns
    -------
    pd.Series | pd.DataFrame
        Crecimiento acumulado partiendo de 1.
    """

    if method == "simple":
        cumulative_growth = (1 + returns).cumprod()

    elif method == "log":
        cumulative_growth = np.exp(returns.cumsum())

    else:
        raise ValueError("method debe ser 'simple' o 'log'.")

    return cumulative_growth


def compute_equity_curve(
    returns: pd.Series | pd.DataFrame,
    initial_capital: float = 10_000.0,
    method: str = "simple"
) -> pd.Series | pd.DataFrame:
    """
    Calcula la curva de equity en unidades monetarias.
    """

    cumulative_growth = compute_cumulative_growth(
        returns=returns,
        method=method
    )

    return initial_capital * cumulative_growth


growth_assets = compute_cumulative_growth(
    asset_returns,
    method=RETURN_METHOD
)

growth_benchmark = compute_cumulative_growth(
    benchmark_returns,
    method=RETURN_METHOD
)

plt.figure(figsize=(12, 6))

for col in growth_assets.columns:
    plt.plot(growth_assets.index, growth_assets[col], label=col, alpha=0.8)

plt.plot(
    growth_benchmark.index,
    growth_benchmark[BENCHMARK_TICKER],
    label=BENCHMARK_TICKER,
    linewidth=2.5,
    linestyle="--"
)

plt.title("Crecimiento acumulado de activos y benchmark")
plt.xlabel("Fecha")
plt.ylabel("Crecimiento de 1 unidad monetaria")
plt.legend()
plt.tight_layout()
plt.show()

# Features del modelo

In [ ]:
# =========================================================
# 14. DESCARGA DEL VIX COMO VARIABLE DE CONTEXTO
# =========================================================

VIX_TICKER = "^VIX"

vix_daily = download_adjusted_prices(
    tickers=[VIX_TICKER],
    start_date=START_DATE,
    end_date=END_DATE
)

vix_daily.columns = ["VIX"]

# Recortamos al periodo común usado en el resto del trabajo
vix_daily = vix_daily.loc[prices_common.index.min():].ffill().dropna()

# Pasamos a frecuencia semanal
vix_weekly = vix_daily.resample(PRICE_FREQ).last().ffill().dropna()

# Alineamos con retornos semanales
vix_weekly = vix_weekly.reindex(asset_returns.index).ffill().dropna()

print("Dimensión VIX semanal:", vix_weekly.shape)
display(vix_weekly.head())
display(vix_weekly.tail())

In [ ]:
# =========================================================
# 15. FUNCIONES AUXILIARES DE FEATURES
# =========================================================

def compute_rsi(prices: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    """
    Calcula el RSI para cada activo.
    """
    delta = prices.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return rsi


def rolling_return(returns: pd.DataFrame, window: int, return_method: str = "simple") -> pd.DataFrame:
    """
    Calcula retorno acumulado rolling.

       Parameters
    ----------
    returns : pd.Series | pd.DataFrame
        Serie o DataFrame de retornos.
    window : int
        Tamaño de ventana.
    method : str
        Tipo de retorno utilizado:
        - "simple": retornos simples.
        - "log": retornos logarítmicos.

    Returns
    -------
    pd.Series | pd.DataFrame
        Crecimiento acumulado partiendo de 1.
    """
    if return_method == "simple":
        return (1 + returns).rolling(window=window).apply(np.prod, raw=True) - 1

    elif return_method == "log":
        return np.exp(returns.rolling(window=window).sum()) - 1

    else:
        raise ValueError("method debe ser 'simple' o 'log'.")



def rolling_volatility(returns: pd.DataFrame, window: int) -> pd.DataFrame:
    """
    Calcula volatilidad rolling.
    """
    return returns.rolling(window=window).std()


def moving_average_ratio(prices: pd.DataFrame, short_window: int, long_window: int) -> pd.DataFrame:
    """
    Cociente entre media móvil corta y media móvil larga.
    """
    ma_short = prices.rolling(short_window).mean()
    ma_long = prices.rolling(long_window).mean()

    return (ma_short / ma_long) - 1


def distance_to_ma(prices: pd.DataFrame, window: int) -> pd.DataFrame:
    """
    Distancia porcentual del precio respecto a su media móvil.
    """
    ma = prices.rolling(window).mean()

    return (prices / ma) - 1

In [ ]:
# =========================================================
# 16. FEATURES POR ACTIVO
# =========================================================

def one_period_return(
    returns: pd.Series | pd.DataFrame,
    return_method: str = "simple"
) -> pd.Series | pd.DataFrame:
    """
    Convierte el retorno de un periodo al formato simple.

    Si los retornos originales son simples, los deja igual.
    Si son logarítmicos, los convierte a simples.
    """

    if return_method == "simple":
        return returns

    elif return_method == "log":
        return np.exp(returns) - 1

    else:
        raise ValueError("return_method debe ser 'simple' o 'log'.")


def build_asset_features(
    prices: pd.DataFrame,
    returns: pd.DataFrame,
    return_method: str = "simple"
) -> pd.DataFrame:
    """
    Construye features por activo.

    Devuelve un DataFrame con columnas MultiIndex:
    nivel 0 = feature
    nivel 1 = ticker
    """
    features = {}

    features["ret_1w"] = one_period_return(
        returns,
        return_method=return_method
    )

    features["ret_4w"] = rolling_return(
        returns,
        window=4,
        return_method=return_method
    )

    features["ret_12w"] = rolling_return(
        returns,
        window=12,
        return_method=return_method
    )

    features["vol_4w"] = rolling_volatility(returns, window=4)
    features["vol_12w"] = rolling_volatility(returns, window=12)

    features["rsi_14"] = compute_rsi(prices, window=14) / 100.0

    features["ma_ratio_4_12"] = moving_average_ratio(
        prices,
        short_window=4,
        long_window=12
    )

    features["dist_ma_12"] = distance_to_ma(
        prices,
        window=12
    )

    feature_df = pd.concat(features, axis=1)
    feature_df = feature_df.sort_index(axis=1)

    return feature_df


asset_features = build_asset_features(
    prices=asset_prices,
    returns=asset_returns,
    return_method= RETURN_METHOD
)

print("Dimensión features por activo:", asset_features.shape)
display(asset_features.head())

In [ ]:
# =========================================================
# 17. FEATURES GLOBALES DE MERCADO
# =========================================================

def build_global_features(
    vix: pd.DataFrame,
    benchmark_returns: pd.DataFrame,
    return_method: str = "simple"
) -> pd.DataFrame:
    """
    Construye variables globales de mercado.
    """
    global_features = pd.DataFrame(index=benchmark_returns.index)

    global_features["vix_level"] = vix["VIX"]
    global_features["vix_ret_1w"] = vix["VIX"].pct_change()
    global_features["acwi_ret_4w"] = rolling_return(
        benchmark_returns[[BENCHMARK_TICKER]],
        window=4,
        return_method=return_method
    )[BENCHMARK_TICKER]

    return global_features


global_features = build_global_features(
    vix=vix_weekly,
    benchmark_returns=benchmark_returns,
    return_method=RETURN_METHOD
)

print("Dimensión features globales:", global_features.shape)
display(global_features.head())

In [ ]:
# =========================================================
# 18. UNIÓN DE FEATURES
# =========================================================

# Aplanamos MultiIndex
asset_features_flat = asset_features.copy()
asset_features_flat.columns = [
    f"{feature}_{ticker}"
    for feature, ticker in asset_features_flat.columns
]

all_features = pd.concat(
    [asset_features_flat, global_features],
    axis=1
)

# Limpieza
all_features = all_features.replace([np.inf, -np.inf], np.nan)
all_features = all_features.dropna()

# Alineamos retornos con features
common_feature_index = all_features.index.intersection(asset_returns.index).intersection(benchmark_returns.index)

all_features = all_features.loc[common_feature_index]
asset_returns_model = asset_returns.loc[common_feature_index]
benchmark_returns_model = benchmark_returns.loc[common_feature_index]

print("Dimensión final de features:", all_features.shape)
print("Dimensión retornos activos alineados:", asset_returns_model.shape)
print("Dimensión benchmark alineado:", benchmark_returns_model.shape)
print("¿NaN en features?:", all_features.isna().sum().sum())

display(all_features.head())

In [ ]:
# =========================================================
# 19. FUNCIÓN DE NORMALIZACIÓN TEMPORAL SIN LEAKAGE
# =========================================================

def fit_transform_train_transform_test(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler]:
    """
    Ajusta StandardScaler solo con train y transforma train/test.
    Evita look-ahead bias.
    """
    scaler = StandardScaler()

    train_scaled = pd.DataFrame(
        scaler.fit_transform(train_features),
        index=train_features.index,
        columns=train_features.columns
    )

    test_scaled = pd.DataFrame(
        scaler.transform(test_features),
        index=test_features.index,
        columns=test_features.columns
    )

    return train_scaled, test_scaled, scaler

In [ ]:
# =========================================================
# 20. RESUMEN DE FEATURES
# =========================================================

feature_summary = pd.DataFrame({
    "feature": all_features.columns,
    "missing_values": all_features.isna().sum().values,
    "mean": all_features.mean().values,
    "std": all_features.std().values,
    "min": all_features.min().values,
    "max": all_features.max().values
})

display(feature_summary.round(4))

In [ ]:
# =========================================================
# 21. COMPROBACIÓN DIMENSIONAL
# =========================================================

n_assets = len(ASSET_TICKERS)
n_asset_features = 8
n_global_features = 3

expected_features = n_assets * n_asset_features + n_global_features

print("Número de activos:", n_assets)
print("Features por activo:", n_asset_features)
print("Features globales:", n_global_features)
print("Features esperadas:", expected_features)
print("Features reales:", all_features.shape[1])

assert all_features.shape[1] == expected_features, "El número de features no coincide con lo esperado."

print("Comprobación superada.")

# Benchmarks: ACWI, Equal Weight y Markowitz

In [ ]:
# =========================================================
# 22. FUNCIONES AUXILIARES PARA BACKTEST
# =========================================================

def portfolio_returns_from_weights(
    returns: pd.DataFrame,
    weights: pd.Series | np.ndarray,
    return_method: str = "simple"
) -> pd.Series:
    """
    Calcula retornos de cartera con pesos fijos.
    """
    if isinstance(weights, np.ndarray):
        weights = pd.Series(weights, index=returns.columns)
    returns_for_backtest = convert_returns_for_backtest(
        returns,
        return_method=return_method
    )

    return returns_for_backtest.dot(weights)


def annualized_return(
    returns: pd.Series,
    periods_per_year: int = 52,
    method: str = "simple"
) -> float:
    """
    Calcula rentabilidad anualizada según el tipo de retorno.
    """

    n_years = len(returns) / periods_per_year

    if method == "simple":
        total_growth = (1 + returns).prod()

    elif method == "log":
        total_growth = np.exp(returns.sum())

    else:
        raise ValueError("method debe ser 'simple' o 'log'.")

    return total_growth ** (1 / n_years) - 1


def annualized_volatility(returns: pd.Series, periods_per_year: int = 52) -> float:
    """
    Volatilidad anualizada.
    """
    return returns.std() * np.sqrt(periods_per_year)


def sharpe_ratio(
    returns: pd.Series,
    risk_free_rate: float = RISK_FREE_RATE,
    periods_per_year: int = 52,
    method: str = "simple"
) -> float:
    """
    Ratio de Sharpe anualizado.

    Se asume que los retornos de estrategia están expresados como retornos simples
    cuando method="simple".
    """

    excess_returns = returns - risk_free_rate / periods_per_year
    vol = excess_returns.std()

    if vol == 0:
        return np.nan

    return excess_returns.mean() / (vol + 1e-8) * np.sqrt(periods_per_year)


def convert_returns_for_backtest(
    returns: pd.Series | pd.DataFrame,
    return_method: str = "simple"
) -> pd.Series | pd.DataFrame:
    """
    Convierte retornos al formato correcto para backtesting.

    En un backtest con pesos de cartera necesitamos retornos simples,
    porque el retorno de cartera se calcula como:

        r_cartera = peso_1 * r_1 + peso_2 * r_2 + ...

    Parameters
    ----------
    returns : pd.Series | pd.DataFrame
        Retornos originales.
    return_method : str
        - "simple": los retornos ya son simples.
        - "log": los retornos son logarítmicos y se convierten a simples.

    Returns
    -------
    pd.Series | pd.DataFrame
        Retornos simples.
    """

    if return_method == "simple":
        return returns

    elif return_method == "log":
        return np.exp(returns) - 1

    else:
        raise ValueError("return_method debe ser 'simple' o 'log'.")

In [ ]:
# =========================================================
# 23. BENCHMARK BUY & HOLD ACWI
# =========================================================

benchmark_bh_returns = convert_returns_for_backtest(
    benchmark_returns_model[BENCHMARK_TICKER],
    return_method=RETURN_METHOD
)

benchmark_bh_returns.name = "BuyHold_ACWI"

benchmark_bh_equity = compute_equity_curve(
    benchmark_bh_returns,
    initial_capital=INITIAL_CAPITAL,
    method=BACKTEST_RETURN_METHOD
)

print("Benchmark Buy & Hold ACWI construido.")
display(benchmark_bh_returns.head())
display(benchmark_bh_equity.tail())

In [ ]:
# =========================================================
# 24. ESTRATEGIA EQUAL WEIGHT
# =========================================================

def backtest_equal_weight(
    returns: pd.DataFrame,
    rebalance_dates: pd.DatetimeIndex,
    transaction_cost: float = 0.001,
    return_method: str = "simple"
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Backtest de estrategia Equal Weight con rebalanceo mensual.

    La estrategia asigna el mismo peso a todos los activos en cada fecha
    de rebalanceo.

    Si returns son log-retornos, se convierten internamente a retornos simples
    para que el cálculo de cartera sea correcto.
    """

    tickers = returns.columns
    n_assets = len(tickers)

    target_weights = pd.Series(
        1 / n_assets,
        index=tickers
    )

    weights_history = []
    portfolio_returns = []

    current_weights = pd.Series(
        0.0,
        index=tickers
    )

    for date in returns.index:

        if date in rebalance_dates:
            turnover = np.abs(target_weights - current_weights).sum()
            cost = transaction_cost * turnover

            current_weights = target_weights.copy()

        else:
            cost = 0.0

        ret_t = returns.loc[date]

        ret_t_for_backtest = convert_returns_for_backtest(
            ret_t,
            return_method=return_method
        )

        pct_ret = float(current_weights.dot(ret_t_for_backtest)) - cost

        portfolio_returns.append(pct_ret)
        weights_history.append(current_weights.copy())

        new_values = current_weights * (1 + ret_t_for_backtest)

        if new_values.sum() > 0:
            current_weights = new_values / new_values.sum()

    returns_series = pd.Series(
        portfolio_returns,
        index=returns.index,
        name="Equal_Weight_Returns"
    )

    weights_df = pd.DataFrame(
        weights_history,
        index=returns.index
    )

    return returns_series, weights_df


equal_weight_returns, equal_weight_weights = backtest_equal_weight(
    returns=asset_returns_model,
    rebalance_dates=rebalance_dates,
    transaction_cost=TC_BPS,
    return_method=RETURN_METHOD
)

equal_weight_equity = compute_equity_curve(
    equal_weight_returns,
    initial_capital=INITIAL_CAPITAL,
    method=BACKTEST_RETURN_METHOD
)

print("Equal Weight construido.")
display(equal_weight_returns.head())
display(equal_weight_weights.head())

In [ ]:
# =========================================================
# 25. FUNCIONES MARKOWITZ MAX SHARPE RESTRINGIDO
# =========================================================

def negative_sharpe_objective(
    weights: np.ndarray,
    expected_returns: np.ndarray,
    cov_matrix: np.ndarray,
    risk_free_rate: float = 0.0
) -> float:
    """
    Objetivo: minimizar el Sharpe negativo.
    """
    port_ret = weights @ expected_returns
    port_vol = np.sqrt(weights @ cov_matrix @ weights)

    if port_vol == 0:
        return 1e6

    return -((port_ret - risk_free_rate) / port_vol)


def max_sharpe_weights(
    returns_window: pd.DataFrame,
    max_weight: float = 0.35,
    allow_short: bool = False,
    risk_free_rate: float = 0.03,
    periods_per_year: int = 52,
    return_method: str = "simple"
) -> pd.Series:
    """
    Calcula pesos Max Sharpe con restricciones.

    Restricciones:
    - La suma de pesos debe ser 1.
    - Si allow_short=False, no se permiten pesos negativos.
    - Cada activo tiene un peso máximo.

    Si returns_window contiene log-retornos, se convierten primero a
    retornos simples antes de estimar medias y covarianzas.
    """

    tickers = returns_window.columns
    n_assets = len(tickers)

    returns_for_optimization = convert_returns_for_backtest(
        returns_window,
        return_method=return_method
    )

    mu = returns_for_optimization.mean().values * periods_per_year
    cov = returns_for_optimization.cov().values * periods_per_year

    x0 = np.repeat(1 / n_assets, n_assets)

    if allow_short:
        bounds = tuple((-max_weight, max_weight) for _ in range(n_assets))
    else:
        bounds = tuple((0.0, max_weight) for _ in range(n_assets))

    constraints = ({
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1.0
    })

    result = minimize(
        fun=negative_sharpe_objective,
        x0=x0,
        args=(mu, cov, risk_free_rate),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 500,
            "ftol": 1e-9,
            "disp": False
        }
    )

    if not result.success:
        weights = x0
    else:
        weights = result.x

    if allow_short:
        weights = np.clip(weights, -max_weight, max_weight)
    else:
        weights = np.clip(weights, 0.0, max_weight)

    weights = weights / weights.sum()

    return pd.Series(weights, index=tickers)

In [ ]:
# =========================================================
# 26. BACKTEST MARKOWITZ MAX SHARPE RESTRINGIDO
# =========================================================

MARKOWITZ_LOOKBACK_WEEKS = 156  # 3 años aprox.

def backtest_markowitz_max_sharpe(
    returns: pd.DataFrame,
    rebalance_dates: pd.DatetimeIndex,
    lookback_weeks: int = 156,
    max_weight: float = 0.35,
    transaction_cost: float = 0.001,
    allow_short: bool = False,
    return_method: str = "simple"
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Backtest de Markowitz Max Sharpe con rebalanceo mensual.

    Si los retornos originales son logarítmicos, se convierten internamente
    a retornos simples para:
    - optimizar pesos,
    - calcular retornos de cartera,
    - actualizar la deriva natural de pesos.
    """

    tickers = returns.columns
    n_assets = len(tickers)

    current_weights = pd.Series(0.0, index=tickers)

    weights_history = []
    portfolio_returns = []

    tc = transaction_cost

    for date in returns.index:

        if date in rebalance_dates:

            hist = returns.loc[:date].iloc[:-1].tail(lookback_weeks)

            if len(hist) >= lookback_weeks:
                target_weights = max_sharpe_weights(
                    returns_window=hist,
                    max_weight=max_weight,
                    allow_short=allow_short,
                    return_method=return_method
                )

            else:
                target_weights = pd.Series(
                    1 / n_assets,
                    index=tickers
                )

            turnover = np.abs(target_weights - current_weights).sum()
            cost = tc * turnover

            current_weights = target_weights.copy()

        else:
            cost = 0.0

        ret_t = returns.loc[date]

        ret_t_for_backtest = convert_returns_for_backtest(
            ret_t,
            return_method=return_method
        )

        pct_ret = float(current_weights.dot(ret_t_for_backtest)) - cost

        portfolio_returns.append(pct_ret)
        weights_history.append(current_weights.copy())

        new_values = current_weights * (1 + ret_t_for_backtest)

        if new_values.sum() > 0:
            current_weights = new_values / new_values.sum()

    returns_series = pd.Series(
        portfolio_returns,
        index=returns.index,
        name="Markowitz_MaxSharpe"
    )

    weights_df = pd.DataFrame(
        weights_history,
        index=returns.index
    )

    return returns_series, weights_df


markowitz_returns, markowitz_weights = backtest_markowitz_max_sharpe(
    returns=asset_returns_model,
    rebalance_dates=rebalance_dates,
    lookback_weeks=MARKOWITZ_LOOKBACK_WEEKS,
    max_weight=MAX_WEIGHT,
    transaction_cost=TC_BPS,
    allow_short=ALLOW_SHORT,
    return_method=RETURN_METHOD
)

markowitz_equity = compute_equity_curve(
    markowitz_returns,
    initial_capital=INITIAL_CAPITAL,
    method=BACKTEST_RETURN_METHOD
)

print("Markowitz Max Sharpe restringido construido.")
display(markowitz_returns.head())
display(markowitz_weights.tail())

In [ ]:
# =========================================================
# 27. COMPARACIÓN INICIAL DE BENCHMARKS
# =========================================================

benchmark_returns_df = pd.concat(
    [
        benchmark_bh_returns,
        equal_weight_returns,
        markowitz_returns
    ],
    axis=1
).dropna()

benchmark_equity_df = compute_equity_curve(
    benchmark_returns_df,
    initial_capital=INITIAL_CAPITAL,
    method=BACKTEST_RETURN_METHOD
)

display(benchmark_returns_df.head())
display(benchmark_equity_df.tail())

In [ ]:
# =========================================================
# 28. MÉTRICAS BÁSICAS DE BENCHMARKS
# =========================================================

def max_drawdown(
    returns: pd.Series,
    method: str = "simple"
) -> float:
    """
    Calcula Maximum Drawdown.
    """

    equity = compute_cumulative_growth(
        returns,
        method=method
    )

    running_max = equity.cummax()
    drawdown = equity / running_max - 1

    return drawdown.min()


def benchmark_metrics_table(
    returns_df: pd.DataFrame,
    periods_per_year: int = 52,
    method: str = "simple"
) -> pd.DataFrame:
    """
    Tabla inicial de métricas.
    """

    rows = []

    for col in returns_df.columns:
        r = returns_df[col].dropna()

        if method == "simple":
            cumulative_return = (1 + r).prod() - 1
        elif method == "log":
            cumulative_return = np.exp(r.sum()) - 1
        else:
            raise ValueError("method debe ser 'simple' o 'log'.")

        rows.append({
            "strategy": col,
            "CAGR": annualized_return(
                r,
                periods_per_year=periods_per_year,
                method=method
            ),
            "Volatilidad anual": annualized_volatility(
                r,
                periods_per_year=periods_per_year
            ),
            "Sharpe": sharpe_ratio(
                r,
                periods_per_year=periods_per_year,
                method=method
            ),
            "Max Drawdown": max_drawdown(
                r,
                method=method
            ),
            "Retorno acumulado": cumulative_return
        })

    return pd.DataFrame(rows).set_index("strategy")


metrics_benchmarks = benchmark_metrics_table(
    benchmark_returns_df,
    method=BACKTEST_RETURN_METHOD
)

display(metrics_benchmarks.round(4))

In [ ]:
# =========================================================
# 29. CURVAS DE EQUITY DE BENCHMARKS
# =========================================================

plt.figure(figsize=(12, 6))

for col in benchmark_equity_df.columns:
    plt.plot(
        benchmark_equity_df.index,
        benchmark_equity_df[col],
        label=col
    )

plt.title("Curvas de equity: benchmarks iniciales")
plt.xlabel("Fecha")
plt.ylabel("Valor de cartera (€)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 30. TURNOVER DE ESTRATEGIAS REBALANCEADAS
# =========================================================

def compute_turnover(weights_df: pd.DataFrame) -> pd.Series:
    """
    Calcula turnover entre periodos consecutivos.
    """
    turnover = weights_df.diff().abs().sum(axis=1)
    return turnover.fillna(0)


equal_weight_turnover = compute_turnover(equal_weight_weights)
markowitz_turnover = compute_turnover(markowitz_weights)

turnover_summary = pd.DataFrame({
    "Equal_Weight": equal_weight_turnover,
    "Markowitz_MaxSharpe": markowitz_turnover
})

annual_turnover = turnover_summary.resample("YE").sum()

print("Turnover anual medio:")
display(annual_turnover.mean().round(4))

display(annual_turnover.tail())

# Dataset temporal para LSTM

In [ ]:
# =========================================================
# 31. PARÁMETROS DEL DATASET LSTM
# =========================================================

LOOKBACK_WEEKS = 52       # 1 año de datos semanales
HORIZON_WEEKS = 4         # horizonte fijo de 4 semanas
N_ASSETS = len(ASSET_TICKERS)
N_FEATURES = all_features.shape[1]

print("Lookback semanas:", LOOKBACK_WEEKS)
print("Horizonte futuro semanas:", HORIZON_WEEKS)
print("Número de activos:", N_ASSETS)
print("Número de features:", N_FEATURES)

In [ ]:
# =========================================================
# 32. CONSTRUCCIÓN DE DATASET TEMPORAL
# =========================================================

def build_lstm_dataset(
    features: pd.DataFrame,
    asset_returns: pd.DataFrame,
    decision_dates: pd.DatetimeIndex,
    lookback_weeks: int = 52,
    horizon_weeks: int = 4,
    return_method: str = "simple"
) -> tuple[np.ndarray, np.ndarray, pd.DatetimeIndex]:
    """
    Construye dataset temporal para LSTM.

    Importante:
    - X contiene features históricas.
    - Y_returns contiene retornos futuros de los activos.
    - Para la loss de cartera, Y_returns debe estar en retornos simples.

    Si asset_returns está en log-retornos, se convierte a retornos simples
    antes de construir Y_returns.

    Returns
    -------
    X : np.ndarray
        Shape: (samples, lookback_weeks, n_features)

    Y_returns : np.ndarray
        Shape: (samples, horizon_weeks, n_assets)

    sample_dates : pd.DatetimeIndex
        Fechas de decisión asociadas a cada muestra.
    """

    common_idx = features.index.intersection(asset_returns.index)

    features = features.loc[common_idx].copy()
    asset_returns = asset_returns.loc[common_idx].copy()

    asset_returns_for_loss = convert_returns_for_backtest(
        asset_returns,
        return_method=return_method
    )

    X_list = []
    Y_list = []
    dates_list = []

    for date in decision_dates:

        if date not in common_idx:
            continue

        loc = common_idx.get_loc(date)

        start_x = loc - lookback_weeks
        end_x = loc

        start_y = loc
        end_y = loc + horizon_weeks

        if start_x < 0:
            continue

        if end_y > len(common_idx):
            continue

        X_window = features.iloc[start_x:end_x].values

        Y_window = asset_returns_for_loss.iloc[start_y:end_y].values

        if X_window.shape != (lookback_weeks, features.shape[1]):
            continue

        if Y_window.shape != (horizon_weeks, asset_returns.shape[1]):
            continue

        X_list.append(X_window)
        Y_list.append(Y_window)
        dates_list.append(date)

    X = np.array(X_list, dtype=np.float32)
    Y_returns = np.array(Y_list, dtype=np.float32)
    sample_dates = pd.DatetimeIndex(dates_list)

    return X, Y_returns, sample_dates


X_raw, Y_returns, sample_dates = build_lstm_dataset(
    features=all_features,
    asset_returns=asset_returns_model,
    decision_dates=rebalance_dates,
    lookback_weeks=LOOKBACK_WEEKS,
    horizon_weeks=HORIZON_WEEKS,
    return_method=RETURN_METHOD
)

print("Shape X_raw:", X_raw.shape)
print("Shape Y_returns:", Y_returns.shape)
print("Número de fechas de muestra:", len(sample_dates))
print("Primera muestra:", sample_dates.min().date())
print("Última muestra:", sample_dates.max().date())

In [ ]:
# =========================================================
# 33. VALIDACIONES DEL DATASET
# =========================================================

assert X_raw.ndim == 3, "X debe ser tridimensional."
assert Y_returns.ndim == 3, "Y_returns debe ser tridimensional."

assert X_raw.shape[0] == Y_returns.shape[0] == len(sample_dates), "Número de muestras inconsistente."
assert X_raw.shape[1] == LOOKBACK_WEEKS, "Lookback incorrecto."
assert X_raw.shape[2] == N_FEATURES, "Número de features incorrecto."
assert Y_returns.shape[1] == HORIZON_WEEKS, "Horizonte incorrecto."
assert Y_returns.shape[2] == N_ASSETS, "Número de activos incorrecto."

assert not np.isnan(X_raw).any(), "Hay NaN en X_raw."
assert not np.isnan(Y_returns).any(), "Hay NaN en Y_returns."

assert not np.isinf(X_raw).any(), "Hay infinitos en X_raw."
assert not np.isinf(Y_returns).any(), "Hay infinitos en Y_returns."

print("Validaciones superadas correctamente.")

In [ ]:
# =========================================================
# 34. DIVISIÓN TEMPORAL TRAIN / VALIDATION / TEST
# =========================================================

def temporal_train_val_test_split(
    X: np.ndarray,
    Y: np.ndarray,
    dates: pd.DatetimeIndex,
    train_ratio: float = 0.60,
    val_ratio: float = 0.20
) -> dict:
    """
    División temporal secuencial.
    """
    n = len(dates)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    split = {
        "X_train": X[:train_end],
        "Y_train": Y[:train_end],
        "dates_train": dates[:train_end],

        "X_val": X[train_end:val_end],
        "Y_val": Y[train_end:val_end],
        "dates_val": dates[train_end:val_end],

        "X_test": X[val_end:],
        "Y_test": Y[val_end:],
        "dates_test": dates[val_end:]
    }

    return split


split_data_raw = temporal_train_val_test_split(
    X=X_raw,
    Y=Y_returns,
    dates=sample_dates,
    train_ratio=0.60,
    val_ratio=0.20
)

print("Train:", split_data_raw["X_train"].shape, split_data_raw["dates_train"][0].date(), "->", split_data_raw["dates_train"][-1].date())
print("Val:", split_data_raw["X_val"].shape, split_data_raw["dates_val"][0].date(), "->", split_data_raw["dates_val"][-1].date())
print("Test:", split_data_raw["X_test"].shape, split_data_raw["dates_test"][0].date(), "->", split_data_raw["dates_test"][-1].date())

In [ ]:
# =========================================================
# 35. NORMALIZACIÓN 3D SIN LEAKAGE
# =========================================================

def scale_lstm_3d_data(
    X_train: np.ndarray,
    X_val: np.ndarray,
    X_test: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray, StandardScaler]:
    """
    Normaliza datos 3D para LSTM ajustando el scaler solo con train.

    X shape: (samples, timesteps, features)
    """
    n_train, t, f = X_train.shape

    scaler = StandardScaler()

    X_train_2d = X_train.reshape(-1, f)
    scaler.fit(X_train_2d)

    def transform(X):
        n = X.shape[0]
        X_2d = X.reshape(-1, f)
        X_scaled = scaler.transform(X_2d)
        return X_scaled.reshape(n, t, f).astype(np.float32)

    X_train_scaled = transform(X_train)
    X_val_scaled = transform(X_val)
    X_test_scaled = transform(X_test)

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler


X_train, X_val, X_test, scaler_lstm = scale_lstm_3d_data(
    X_train=split_data_raw["X_train"],
    X_val=split_data_raw["X_val"],
    X_test=split_data_raw["X_test"]
)

Y_train = split_data_raw["Y_train"]
Y_val = split_data_raw["Y_val"]
Y_test = split_data_raw["Y_test"]

dates_train = split_data_raw["dates_train"]
dates_val = split_data_raw["dates_val"]
dates_test = split_data_raw["dates_test"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("Y_train:", Y_train.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)

print("Media train escalado aprox:", X_train.mean().round(4))
print("Std train escalado aprox:", X_train.std().round(4))

In [ ]:
# =========================================================
# 36. COMPROBACIÓN FINAL DATASET LSTM
# =========================================================

assert not np.isnan(X_train).any()
assert not np.isnan(X_val).any()
assert not np.isnan(X_test).any()

assert not np.isnan(Y_train).any()
assert not np.isnan(Y_val).any()
assert not np.isnan(Y_test).any()

print("Dataset LSTM preparado correctamente.")
print("Entrada de una muestra:", X_train[0].shape)
print("Retornos futuros de una muestra:", Y_train[0].shape)
print("Fecha asociada primera muestra train:", dates_train[0].date())

# Arquitectura LSTM

In [ ]:
# =========================================================
# 37. MODELO LSTM GENERADOR DE PESOS
# =========================================================

def build_lstm_weight_model(
    lookback_weeks: int,
    n_features: int,
    n_assets: int,
    lstm_units_1: int = 32,
    lstm_units_2: int = 16,
    dense_units: int = 16,
    dropout_rate: float = 0.20,
    l2_reg: float = 1e-4
) -> keras.Model:
    """
    Construye una red LSTM end-to-end que genera directamente
    pesos de cartera.

    La salida softmax garantiza:
    - pesos no negativos
    - suma de pesos igual a 1
    """

    inputs = keras.Input(
        shape=(lookback_weeks, n_features),
        name="input_features"
    )

    x = layers.LSTM(
        lstm_units_1,
        return_sequences=True,
        kernel_regularizer=regularizers.l2(l2_reg),
        name="lstm_1"
    )(inputs)

    x = layers.Dropout(dropout_rate, name="dropout_1")(x)

    x = layers.LSTM(
        lstm_units_2,
        return_sequences=False,
        kernel_regularizer=regularizers.l2(l2_reg),
        name="lstm_2"
    )(x)

    x = layers.Dropout(dropout_rate, name="dropout_2")(x)

    x = layers.Dense(
        dense_units,
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="dense_hidden"
    )(x)

    raw_weights = layers.Dense(
        n_assets,
        activation=None,
        name="raw_weights"
    )(x)

    portfolio_weights = layers.Softmax(
        name="portfolio_weights"
    )(raw_weights)

    model = keras.Model(
        inputs=inputs,
        outputs=portfolio_weights,
        name="LSTM_EndToEnd_Portfolio"
    )

    return model


lstm_model = build_lstm_weight_model(
    lookback_weeks=LOOKBACK_WEEKS,
    n_features=N_FEATURES,
    n_assets=N_ASSETS
)

lstm_model.summary()

In [ ]:
# =========================================================
# 38. COMPROBACIÓN DE PESOS GENERADOS
# =========================================================

sample_input = X_train[:5]

sample_weights = lstm_model.predict(sample_input, verbose=0)

weights_check = pd.DataFrame(
    sample_weights,
    columns=ASSET_TICKERS,
    index=dates_train[:5]
)

display(weights_check.round(4))

print("Suma de pesos por muestra:")
print(weights_check.sum(axis=1).round(6))

print("Peso mínimo:", round(weights_check.min().min(), 6))
print("Peso máximo:", round(weights_check.max().max(), 6))

In [ ]:
# =========================================================
# 39. FUNCIÓN PARA LIMITAR PESO MÁXIMO
# =========================================================

def cap_and_renormalize_weights(
    weights: np.ndarray,
    max_weight: float = 0.35,
    tolerance: float = 1e-8,
    max_iter: int = 100
) -> np.ndarray:
    """
    Aplica un límite máximo de peso por activo y renormaliza
    redistribuyendo el exceso solo entre activos que aún no alcanzan el límite.

    Parameters
    ----------
    weights : np.ndarray
        Vector de pesos.
    max_weight : float
        Peso máximo permitido por activo.

    Returns
    -------
    np.ndarray
        Pesos ajustados.
    """
    w = np.asarray(weights, dtype=float).copy()

    if np.any(w < -tolerance):
        raise ValueError("Los pesos contienen valores negativos.")

    w = np.clip(w, 0, None)

    if w.sum() <= tolerance:
        return np.repeat(1 / len(w), len(w))

    w = w / w.sum()

    capped = np.zeros_like(w, dtype=bool)

    for _ in range(max_iter):
        above = w > max_weight

        if not above.any():
            break

        capped = capped | above
        excess = np.sum(w[above] - max_weight)
        w[above] = max_weight

        free = ~capped

        if not free.any():
            break

        free_sum = w[free].sum()

        if free_sum <= tolerance:
            break

        w[free] += excess * (w[free] / free_sum)

    # Corrección final: asegurar estrictamente el cap
    w = np.minimum(w, max_weight)

    # Redistribuir cualquier diferencia residual sin romper el cap
    residual = 1.0 - w.sum()

    for _ in range(max_iter):
        if abs(residual) <= tolerance:
            break

        free = w < (max_weight - tolerance)

        if not free.any():
            break

        capacity = max_weight - w[free]
        add = residual * (capacity / capacity.sum())
        w[free] += add

        w = np.minimum(w, max_weight)
        residual = 1.0 - w.sum()

    w = w / w.sum()

    # Última protección frente a redondeos
    if w.max() > max_weight + 1e-8:
        w = np.minimum(w, max_weight)
        w = w / w.sum()

    return w

In [ ]:
# =========================================================
# 40. TEST DE CAPADO Y RENORMALIZACIÓN
# =========================================================

test_w = np.array([0.60, 0.10, 0.10, 0.05, 0.05, 0.05, 0.05])

adjusted_w = cap_and_renormalize_weights(
    weights=test_w,
    max_weight=MAX_WEIGHT
)

pd.DataFrame(
    {
        "peso_original": test_w,
        "peso_ajustado": adjusted_w
    },
    index=ASSET_TICKERS
).round(4)

In [ ]:
# =========================================================
# 41. RESUMEN DEL DISEÑO DEL MODELO
# =========================================================

model_design_summary = pd.DataFrame({
    "Elemento": [
        "Tipo de modelo",
        "Entrada",
        "Lookback",
        "Número de features",
        "Número de activos",
        "Salida",
        "Restricción de suma",
        "Restricción long-only",
        "Peso máximo por activo",
        "Regularización",
        "Dropout"
    ],
    "Valor": [
        "LSTM end-to-end",
        f"({LOOKBACK_WEEKS}, {N_FEATURES})",
        f"{LOOKBACK_WEEKS} semanas",
        N_FEATURES,
        N_ASSETS,
        "Vector de pesos",
        "Softmax",
        "Softmax",
        MAX_WEIGHT,
        "L2",
        "0.20"
    ]
})

display(model_design_summary)

# Función de pérdida con Sharpe + turnover

In [ ]:
# =========================================================
# 42. FUNCIÓN DE PÉRDIDA: SHARPE RATIO
# =========================================================

def sharpe_loss(y_true, y_pred):
    """
    Loss basada en Sharpe Ratio.

    Importante:
    y_true debe contener retornos simples, no log-retornos.

    Parameters
    ----------
    y_true : tensor
        Retornos futuros simples de los activos.
        Shape: (batch, horizon, assets)

    y_pred : tensor
        Pesos de cartera generados por la red.
        Shape: (batch, assets)

    Returns
    -------
    tensor
        Sharpe negativo. Se devuelve en negativo porque Keras minimiza la loss.
    """

    portfolio_returns = tf.reduce_sum(
        y_pred[:, None, :] * y_true,
        axis=2
    )  # Resultado: (batch, horizon)

    # Aplanamos todos los retornos del batch y del horizonte.
    portfolio_returns = tf.reshape(portfolio_returns, [-1])

    mean_return = tf.reduce_mean(portfolio_returns)
    std_return = tf.math.reduce_std(portfolio_returns) + 1e-8

    sharpe = mean_return / std_return

    # Devolvemos Sharpe negativo porque el entrenamiento minimiza la loss.
    return -sharpe

In [ ]:
# =========================================================
# 43. PENALIZACIÓN DE TURNOVER
# =========================================================

def turnover_penalty(y_pred):
    """
    Penaliza cambios entre pesos consecutivos en el batch.

    y_pred: (batch, assets)
    """
    w_t = y_pred[1:]
    w_prev = y_pred[:-1]

    turnover = tf.reduce_sum(tf.abs(w_t - w_prev), axis=1)

    return tf.reduce_mean(turnover)

In [ ]:
# =========================================================
# 44. FUNCIÓN DE PÉRDIDA FINAL
# =========================================================

LAMBDA_TURNOVER = 0.10  

def sharpe_turnover_loss(y_true, y_pred):
    """
    Loss final:
    - Maximiza Sharpe
    - Penaliza turnover
    """

    sharpe_component = sharpe_loss(y_true, y_pred)
    turnover_component = turnover_penalty(y_pred)

    loss = sharpe_component + LAMBDA_TURNOVER * turnover_component

    return loss

In [ ]:
# =========================================================
# 45. COMPILACIÓN DEL MODELO
# =========================================================

optimizer = keras.optimizers.Adam(
    learning_rate=1e-3
)

lstm_model.compile(
    optimizer=optimizer,
    loss=sharpe_turnover_loss
)

print("Modelo compilado correctamente.")

In [ ]:
# =========================================================
# 46. TEST DE LA FUNCIÓN DE PÉRDIDA
# =========================================================

X_test_sample = X_train[:32]
Y_test_sample = Y_train[:32]

weights_pred = lstm_model(X_test_sample)

loss_value = sharpe_turnover_loss(
    tf.constant(Y_test_sample),
    weights_pred
)

print("Loss inicial:", float(loss_value))

In [ ]:
# =========================================================
# 47. DESGLOSE DE LA LOSS
# =========================================================

def debug_loss_components(y_true, y_pred):

    portfolio_returns = tf.reduce_sum(
        y_pred[:, None, :] * y_true,
        axis=2
    )

    portfolio_returns = tf.reshape(portfolio_returns, [-1])

    mean_return = tf.reduce_mean(portfolio_returns)
    std_return = tf.math.reduce_std(portfolio_returns)

    sharpe = mean_return / (std_return + 1e-8)

    turnover = turnover_penalty(y_pred)

    print("Mean return:", float(mean_return))
    print("Std return:", float(std_return))
    print("Sharpe:", float(sharpe))
    print("Turnover:", float(turnover))


debug_loss_components(
    tf.constant(Y_test_sample),
    weights_pred
)

# Walk-forward

In [ ]:
# =========================================================
# 48. PARÁMETROS WALK-FORWARD LSTM
# =========================================================

TRAIN_MIN_SAMPLES = 80      # mínimo de muestras para empezar a entrenar
VAL_SAMPLES = 24            # aprox. 2 años de validación mensual
RETRAIN_EVERY = 12          # reentrenar cada 12 muestras/meses

EPOCHS = 100
BATCH_SIZE = 16
PATIENCE = 12

LSTM_TC_BPS = TC_BPS
LSTM_MAX_WEIGHT = MAX_WEIGHT

print("Parámetros walk-forward definidos.")

In [ ]:
# =========================================================
# 49. CREACIÓN DE MODELO COMPILADO
# =========================================================

def create_compiled_lstm_model() -> keras.Model:
    """
    Crea y compila un modelo LSTM nuevo.
    """
    model = build_lstm_weight_model(
        lookback_weeks=LOOKBACK_WEEKS,
        n_features=N_FEATURES,
        n_assets=N_ASSETS,
        lstm_units_1=32,
        lstm_units_2=16,
        dense_units=16,
        dropout_rate=0.20,
        l2_reg=1e-4
    )

    optimizer = keras.optimizers.Adam(learning_rate=1e-3)

    model.compile(
        optimizer=optimizer,
        loss=sharpe_turnover_loss
    )

    return model

In [ ]:
# =========================================================
# 50. ESCALADO WALK-FORWARD SIN LEAKAGE
# =========================================================

def scale_walk_forward_window(
    X_train_window: np.ndarray,
    X_val_window: np.ndarray,
    X_test_window: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray, StandardScaler]:
    """
    Ajusta el scaler SOLO con train y transforma train/val/test.
    """
    n_train, t, f = X_train_window.shape

    scaler = StandardScaler()
    scaler.fit(X_train_window.reshape(-1, f))

    def transform(X):
        n = X.shape[0]
        return scaler.transform(X.reshape(-1, f)).reshape(n, t, f).astype(np.float32)

    X_train_scaled = transform(X_train_window)
    X_val_scaled = transform(X_val_window)
    X_test_scaled = transform(X_test_window)

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

In [ ]:
# =========================================================
# 51. WALK-FORWARD TRAINING Y PREDICCIÓN DE PESOS
# =========================================================

def walk_forward_lstm_predictions(
    X: np.ndarray,
    Y: np.ndarray,
    dates: pd.DatetimeIndex,
    train_min_samples: int = 80,
    val_samples: int = 24,
    retrain_every: int = 12,
    epochs: int = 100,
    batch_size: int = 16,
    patience: int = 12,
    verbose: int = 0
) -> tuple[pd.DataFrame, dict]:
    """
    Entrena modelos LSTM en esquema walk-forward y genera pesos fuera de muestra.

    Para cada bloque de test:
    - Train: desde inicio hasta antes de validation.
    - Validation: últimas val_samples antes del test.
    - Test: bloque siguiente hasta próximo reentrenamiento.

    Returns
    -------
    weights_df : pd.DataFrame
        Pesos generados fuera de muestra.
    training_info : dict
        Información de entrenamiento por ventana.
    """
    all_weights = []
    all_dates = []
    training_info = {}

    n_samples = len(dates)
    start_test = train_min_samples + val_samples

    model_counter = 0

    for test_start in range(start_test, n_samples, retrain_every):

        test_end = min(test_start + retrain_every, n_samples)

        train_end = test_start - val_samples
        val_start = train_end
        val_end = test_start

        X_train_w = X[:train_end]
        Y_train_w = Y[:train_end]

        X_val_w = X[val_start:val_end]
        Y_val_w = Y[val_start:val_end]

        X_test_w = X[test_start:test_end]
        test_dates_w = dates[test_start:test_end]

        if len(X_train_w) < train_min_samples or len(X_val_w) == 0 or len(X_test_w) == 0:
            continue

        X_train_s, X_val_s, X_test_s, scaler = scale_walk_forward_window(
            X_train_window=X_train_w,
            X_val_window=X_val_w,
            X_test_window=X_test_w
        )

        model = create_compiled_lstm_model()

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=patience,
                restore_best_weights=True
            )
        ]

        history = model.fit(
            X_train_s,
            Y_train_w,
            validation_data=(X_val_s, Y_val_w),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=False,
            callbacks=callbacks,
            verbose=verbose
        )

        pred_weights = model.predict(X_test_s, verbose=0)

        # Aplicamos capado y renormalización ex-post
        pred_weights_capped = np.vstack([
            cap_and_renormalize_weights(w, max_weight=LSTM_MAX_WEIGHT)
            for w in pred_weights
        ])

        all_weights.append(pred_weights_capped)
        all_dates.extend(test_dates_w)

        training_info[model_counter] = {
            "train_start": dates[0],
            "train_end": dates[train_end - 1],
            "val_start": dates[val_start],
            "val_end": dates[val_end - 1],
            "test_start": test_dates_w[0],
            "test_end": test_dates_w[-1],
            "epochs_trained": len(history.history["loss"]),
            "final_train_loss": history.history["loss"][-1],
            "final_val_loss": history.history["val_loss"][-1]
        }

        print(
            f"Modelo {model_counter}: "
            f"train {dates[0].date()} -> {dates[train_end - 1].date()} | "
            f"val {dates[val_start].date()} -> {dates[val_end - 1].date()} | "
            f"test {test_dates_w[0].date()} -> {test_dates_w[-1].date()} | "
            f"epochs {len(history.history['loss'])}"
        )

        model_counter += 1

    if len(all_weights) == 0:
        raise ValueError("No se generaron pesos. Revisa TRAIN_MIN_SAMPLES, VAL_SAMPLES o número de muestras.")

    weights_array = np.vstack(all_weights)

    weights_df = pd.DataFrame(
        weights_array,
        index=pd.DatetimeIndex(all_dates),
        columns=ASSET_TICKERS
    )

    return weights_df, training_info


lstm_weights_oos, lstm_training_info = walk_forward_lstm_predictions(
    X=X_raw,
    Y=Y_returns,
    dates=sample_dates,
    train_min_samples=TRAIN_MIN_SAMPLES,
    val_samples=VAL_SAMPLES,
    retrain_every=RETRAIN_EVERY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE,
    verbose=0
)

print("\nPesos LSTM OOS generados:")
display(lstm_weights_oos.head())
display(lstm_weights_oos.tail())

print("Shape pesos:", lstm_weights_oos.shape)
print("Suma mínima de pesos:", lstm_weights_oos.sum(axis=1).min())
print("Suma máxima de pesos:", lstm_weights_oos.sum(axis=1).max())
print("Peso máximo observado:", lstm_weights_oos.max().max())

In [ ]:
# =========================================================
# 52. RESUMEN DEL ENTRENAMIENTO WALK-FORWARD
# =========================================================

training_summary = pd.DataFrame(lstm_training_info).T

display(training_summary)

print("Número de modelos entrenados:", len(training_summary))
print("Épocas medias entrenadas:", training_summary["epochs_trained"].mean())

In [ ]:
# =========================================================
# 53. BACKTEST LSTM CON PESOS WALK-FORWARD
# =========================================================

def backtest_lstm_with_monthly_weights(
    returns: pd.DataFrame,
    weights_monthly: pd.DataFrame,
    transaction_cost: float = 0.001,
    return_method: str = "simple"
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Backtest de estrategia LSTM usando pesos generados en fechas mensuales.

    Los pesos se mantienen constantes hasta el siguiente rebalanceo,
    con deriva natural de mercado semanal.

    Si los retornos originales son logarítmicos, se convierten internamente
    a retornos simples para calcular correctamente:
    - retorno de cartera,
    - costes de transacción,
    - deriva de pesos.
    """

    tickers = returns.columns

    weights_monthly = weights_monthly.copy()
    weights_monthly = weights_monthly.sort_index()

    returns_bt = returns.loc[
        returns.index >= weights_monthly.index.min()
    ].copy()

    current_weights = pd.Series(0.0, index=tickers)

    portfolio_returns = []
    weights_history = []

    tc = transaction_cost

    for date in returns_bt.index:

        if date in weights_monthly.index:
            target_weights = weights_monthly.loc[date]

            turnover = np.abs(target_weights - current_weights).sum()
            cost = tc * turnover

            current_weights = target_weights.copy()

        else:
            cost = 0.0

        ret_t = returns_bt.loc[date]

        ret_t_for_backtest = convert_returns_for_backtest(
            ret_t,
            return_method=return_method
        )

        pct_ret = float(current_weights.dot(ret_t_for_backtest)) - cost

        portfolio_returns.append(pct_ret)
        weights_history.append(current_weights.copy())

        new_values = current_weights * (1 + ret_t_for_backtest)

        if new_values.sum() > 0:
            current_weights = new_values / new_values.sum()

    returns_series = pd.Series(
        portfolio_returns,
        index=returns_bt.index,
        name="LSTM_EndToEnd"
    )

    weights_weekly_df = pd.DataFrame(
        weights_history,
        index=returns_bt.index
    )

    return returns_series, weights_weekly_df


lstm_returns, lstm_weights_weekly = backtest_lstm_with_monthly_weights(
    returns=asset_returns_model,
    weights_monthly=lstm_weights_oos,
    transaction_cost=LSTM_TC_BPS,
    return_method=RETURN_METHOD
)

lstm_equity = compute_equity_curve(
    lstm_returns,
    initial_capital=INITIAL_CAPITAL,
    method=BACKTEST_RETURN_METHOD
)

print("Backtest LSTM construido.")
display(lstm_returns.head())
display(lstm_weights_weekly.head())
display(lstm_equity.tail())

In [ ]:
# =========================================================
# 54. VALIDACIÓN DE RESTRICCIONES LSTM
# =========================================================

print("Suma pesos mensuales mínima:", lstm_weights_oos.sum(axis=1).min())
print("Suma pesos mensuales máxima:", lstm_weights_oos.sum(axis=1).max())
print("Peso mínimo mensual:", lstm_weights_oos.min().min())
print("Peso máximo mensual:", lstm_weights_oos.max().max())

assert np.allclose(lstm_weights_oos.sum(axis=1), 1.0, atol=1e-6)
assert (lstm_weights_oos.values >= -1e-8).all()
assert (lstm_weights_oos.values <= LSTM_MAX_WEIGHT + 1e-6).all()

print("Restricciones de pesos LSTM superadas.")

In [ ]:
# =========================================================
# 55. TURNOVER LSTM
# =========================================================

lstm_turnover_monthly = compute_turnover(lstm_weights_oos)
lstm_turnover_annual = lstm_turnover_monthly.resample("YE").sum()

print("Turnover mensual medio LSTM:", round(lstm_turnover_monthly.mean(), 4))
print("Turnover anual medio LSTM:", round(lstm_turnover_annual.mean(), 4))
print("Umbral anual máximo definido:", MAX_TURNOVER_YEAR)

display(lstm_turnover_annual.round(4))

if lstm_turnover_annual.mean() <= MAX_TURNOVER_YEAR:
    print("Turnover medio anual dentro del umbral definido.")
else:
    print("Aviso: turnover medio anual superior al umbral definido.")

In [ ]:
# =========================================================
# 56. COMPARACIÓN EN PERIODO COMÚN
# =========================================================

all_strategy_returns = pd.concat(
    [
        benchmark_bh_returns,
        equal_weight_returns,
        markowitz_returns,
        lstm_returns
    ],
    axis=1
).dropna()

all_strategy_equity = compute_equity_curve(
    all_strategy_returns,
    initial_capital=INITIAL_CAPITAL,
    method=BACKTEST_RETURN_METHOD
)

print("Periodo común de comparación:")
print(all_strategy_returns.index.min().date(), "->", all_strategy_returns.index.max().date())

display(all_strategy_returns.head())
display(all_strategy_equity.tail())

In [ ]:
# =========================================================
# 57. MÉTRICAS PRELIMINARES TODAS LAS ESTRATEGIAS
# =========================================================

metrics_all = benchmark_metrics_table(
    all_strategy_returns,
    method=BACKTEST_RETURN_METHOD
)

display(metrics_all.round(4))

In [ ]:
# =========================================================
# 58. CURVAS DE EQUITY COMPARADAS
# =========================================================
normalized = all_strategy_equity / all_strategy_equity.iloc[0] * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)

plots = [
    {
        "data": all_strategy_equity,
        "title": "Evolución del valor de la cartera (€)",
        "ylabel": "Valor (€)"
    },
    {
        "data": np.log(all_strategy_equity),
        "title": "Evolución logarítmica de la cartera",
        "ylabel": "Log(valor)"
    },
    {
        "data": normalized,
        "title": "Rentabilidad acumulada normalizada (base 100)",
        "ylabel": "Índice (100 = inicio)"
    }
]

for ax, plot_cfg in zip(axes, plots):
    for col in plot_cfg["data"].columns:
        ax.plot(
            plot_cfg["data"].index,
            plot_cfg["data"][col],
            label=col,
            linewidth=2
        )

    ax.set_title(plot_cfg["title"])
    ax.set_ylabel(plot_cfg["ylabel"])
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Fecha")

plt.tight_layout()
plt.show()

# Backtest y métricas

In [ ]:
# =========================================================
# 59. MÉTRICAS COMPLETAS
# =========================================================

def compute_cagr(
    returns: pd.Series,
    method: str = "simple"
) -> float:
    """
    Calcula CAGR según el tipo de retorno.
    """

    n_years = len(returns) / 52

    if method == "simple":
        total_growth = (1 + returns).prod()

    elif method == "log":
        total_growth = np.exp(returns.sum())

    else:
        raise ValueError("method debe ser 'simple' o 'log'.")

    return total_growth ** (1 / n_years) - 1


def compute_volatility(returns: pd.Series) -> float:
    return returns.std() * np.sqrt(52)


def compute_sharpe(returns: pd.Series, rf: float = RISK_FREE_RATE) -> float:
    """
    Calcula Sharpe anualizado usando retornos simples periódicos.
    """

    excess = returns - rf / 52

    return excess.mean() / (excess.std() + 1e-8) * np.sqrt(52)


def compute_max_drawdown(
    returns: pd.Series,
    method: str = "simple"
) -> float:
    equity = compute_cumulative_growth(
        returns,
        method=method
    )

    peak = equity.cummax()
    drawdown = (equity / peak) - 1

    return drawdown.min()


def compute_tracking_error(strategy: pd.Series, benchmark: pd.Series) -> float:
    diff = strategy - benchmark
    return diff.std() * np.sqrt(52)


def compute_information_ratio(strategy: pd.Series, benchmark: pd.Series) -> float:
    diff = strategy - benchmark
    return diff.mean() / (diff.std() + 1e-8) * np.sqrt(52)

In [ ]:
# =========================================================
# 60. TABLA FINAL DE MÉTRICAS
# =========================================================

def full_metrics_table(
    returns_df: pd.DataFrame,
    benchmark_col: str,
    method: str = "simple"
) -> pd.DataFrame:

    metrics = {}

    benchmark = returns_df[benchmark_col]

    for col in returns_df.columns:

        r = returns_df[col]

        metrics[col] = {
            "CAGR": compute_cagr(
                r,
                method=method
            ),
            "Volatilidad": compute_volatility(r),
            "Sharpe": compute_sharpe(r),
            "Max Drawdown": compute_max_drawdown(
                r,
                method=method
            ),
            "Tracking Error": compute_tracking_error(r, benchmark),
            "Information Ratio": compute_information_ratio(r, benchmark)
        }

    return pd.DataFrame(metrics).T


metrics_final = full_metrics_table(
    all_strategy_returns,
    benchmark_col="BuyHold_ACWI",
    method=BACKTEST_RETURN_METHOD
)

display(metrics_final.round(4))

In [ ]:
# =========================================================
# 61. DRAWDOWNS
# =========================================================

def compute_drawdown_series(
    returns: pd.Series,
    method: str = "simple"
) -> pd.Series:
    equity = compute_cumulative_growth(
        returns,
        method=method
    )

    peak = equity.cummax()

    return (equity / peak) - 1


drawdowns = pd.DataFrame({
    col: compute_drawdown_series(
        all_strategy_returns[col],
        method=BACKTEST_RETURN_METHOD
    )
    for col in all_strategy_returns.columns
})

display(drawdowns.tail())

In [ ]:
# =========================================================
# 62. GRÁFICO DE DRAWDOWN
# =========================================================

plt.figure(figsize=(12, 6))

for col in drawdowns.columns:
    plt.plot(drawdowns.index, drawdowns[col], label=col)

plt.title("Drawdown de las estrategias")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 63. ANÁLISIS EN PERIODOS DE CRISIS
# =========================================================

crisis_periods = {
    "2008 Crisis": ("2007-01-01", "2009-12-31"),
    "COVID": ("2020-01-01", "2021-12-31")
}

crisis_results = {}

for name, (start, end) in crisis_periods.items():

    subset = all_strategy_returns.loc[start:end]

    if len(subset) > 0:
        crisis_results[name] = full_metrics_table(
            subset,
            benchmark_col="BuyHold_ACWI",
            method=BACKTEST_RETURN_METHOD
        )

for k, v in crisis_results.items():
    print(f"\n===== {k} =====")
    display(v.round(4))

In [ ]:
# =========================================================
# 64. EVOLUCIÓN DE PESOS LSTM
# =========================================================
plt.figure(figsize=(12, 6))

for col in lstm_weights_oos.columns:
    plt.plot(lstm_weights_oos.index, lstm_weights_oos[col], label=col)

plt.title("Evolución de pesos LSTM (mensual)")
plt.xlabel("Fecha")
plt.ylabel("Peso")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Copia de los pesos generados por el modelo
weights_plot = lstm_weights_oos.copy()

# Aseguramos que las columnas siguen el orden
weights_plot = weights_plot[ASSET_TICKERS]

# Comprobación de que los pesos suman aprox. 1 en cada fecha
assert np.allclose(weights_plot.sum(axis=1), 1.0, atol=1e-6), "Los pesos no suman 1."

# Creación del gráfico
fig, ax = plt.subplots(figsize=(15, 7))

ax.stackplot(
    weights_plot.index,
    [weights_plot[col] for col in weights_plot.columns],
    labels=weights_plot.columns,
    alpha=0.9
)

ax.set_title("Evolución mensual de los pesos del modelo LSTM", fontsize=14)
ax.set_xlabel("Fecha")
ax.set_ylabel("Asignación de la cartera")
ax.set_ylim(0, 1)

# Formato porcentual del eje Y
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Rejilla horizontal para facilitar lectura
ax.grid(axis="y", alpha=0.3)

# Leyenda fuera del gráfico
ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    title="Activos",
    frameon=True
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 65. TURNOVER COMPARATIVO
# =========================================================

turnover_summary = pd.DataFrame({
    "Equal Weight": equal_weight_turnover.mean(),
    "Markowitz": markowitz_turnover.mean(),
    "LSTM": lstm_turnover_monthly.mean()
}, index=["Turnover medio mensual"]).T

display(turnover_summary.round(4))

# Gráficos y análisis

In [ ]:
# =========================================================
# 66. ROLLING SHARPE
# =========================================================

window = 52

rolling_excess_returns = all_strategy_returns - RISK_FREE_RATE / 52

rolling_sharpe = rolling_excess_returns.rolling(window).mean() / (
    rolling_excess_returns.rolling(window).std() + 1e-8
) * np.sqrt(52)

plt.figure(figsize=(12, 6))

for col in rolling_sharpe.columns:
    plt.plot(rolling_sharpe.index, rolling_sharpe[col], label=col)

plt.title("Rolling Sharpe (52 semanas)")
plt.xlabel("Fecha")
plt.ylabel("Sharpe")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 67. ROLLING VOLATILITY
# =========================================================

rolling_vol = all_strategy_returns.rolling(window).std() * np.sqrt(52)

plt.figure(figsize=(12, 6))

for col in rolling_vol.columns:
    plt.plot(rolling_vol.index, rolling_vol[col], label=col)

plt.title("Volatilidad rolling (52 semanas)")
plt.xlabel("Fecha")
plt.ylabel("Volatilidad")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 68. GRÁFICO DE MÉTRICAS
# =========================================================

metrics_plot = metrics_final.copy()

metrics_plot[["CAGR", "Sharpe", "Information Ratio"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Comparación de métricas clave")
plt.xticks(rotation=45)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 69. EXPOSICIÓN RIESGO VS DEFENSIVO
# =========================================================

equity_assets = ["SPY", "VGK", "EWJ", "VWO", "XLK", "XLE"]
gold_asset = ["GLD"]

equity_exposure = lstm_weights_oos[equity_assets].sum(axis=1)
gold_exposure = lstm_weights_oos[gold_asset].sum(axis=1)

plt.figure(figsize=(12, 6))

plt.plot(equity_exposure.index, equity_exposure, label="Renta variable")
plt.plot(gold_exposure.index, gold_exposure, label="Oro (GLD)")

plt.title("Exposición dinámica del modelo LSTM")
plt.xlabel("Fecha")
plt.ylabel("Peso")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()